In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu118

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [82]:
import torch.nn as nn
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import pandas as pd
from collections import Counter
import random
import ast

In [83]:
df = pd.read_csv("num_dataset_10.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [84]:
sos_val = 10
eos_val = 11
pad_val = 12
vocab_size = 13

In [85]:
inp_tensors = [torch.tensor([sos_val] + ast.literal_eval(s) + [eos_val], dtype=torch.long) for s in df['source']]
out_tensors = [torch.tensor([sos_val] + ast.literal_eval(s) + [eos_val], dtype=torch.long) for s in df['target']]

In [86]:
inp_padded = nn.utils.rnn.pad_sequence(inp_tensors, batch_first=True, padding_value=pad_val)
out_padded = nn.utils.rnn.pad_sequence(out_tensors, batch_first=True, padding_value=pad_val)

In [87]:
dataset = TensorDataset(inp_padded, out_padded)

#80, 10, 10 split for Train, Val, Test
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset,  batch_size=128, shuffle=False)

In [88]:
class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size, num_layers):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        #Expands a token to dim embed_size, based on learned embeddings
        embedding = self.dropout(self.embedding(x))

        #Output is all the hidden states, hidden is just final vector which is what we want
        output, hidden = self.gru(embedding)

        return hidden

In [89]:
class Decoder(nn.Module):
    #Input and output sizes are the same here b/c both use same vocab space
    def __init__(self, input_size, embed_size, hidden_size, output_size, num_layers):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(input_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.2)

        #Fully connected layer, maps dim of lstm outputs to vocab dim
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        #Converts shape of x from (batch_size) to (1, batch_size)
        x = x.unsqueeze(1)
        #Converts shape now to (1, batch_size, embed_size)
        embedding = self.dropout(self.embedding(x))

        #Outputs = hidden here, since we're doing a sequence of length 1
        outputs, hidden = self.gru(embedding, hidden)

        # predictions = self.fc(self.dropout_out(outputs))
        predictions = self.fc(outputs)
        predictions = predictions.squeeze(1)

        return predictions, hidden

In [90]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target, teacher_force_ratio):
        batch_size = source.shape[0]
        target_len = source.shape[1]

        #Tensor of zeros with shape (batch_size, target_len, vocab_size), will gradually be filled in by the decoder
        #BN: Vocab_size != Embedding_Size
        outputs = torch.zeros(batch_size, target_len, vocab_size).to(device)

        #Context vector
        hidden = self.encoder(source)

        #Effectively the <sos> token
        # x = target[:, 0]
        x = torch.full((batch_size,), sos_val, dtype=torch.long).to(device)

        for t in range(1, target_len):
            output, hidden = self.decoder(x, hidden)
            #Fill in the outputs variable
            outputs[:, t] = output

            #Chooses the one with the highest probability
            best_guess = output.argmax(1)

            #Using random to ensure teacher forcing is applied only a certain percentage of the time
            if random.random() < teacher_force_ratio:
                x = target[:, t]
            else:
                x = best_guess

        return outputs


In [91]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, loader, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for src_batch, trg_batch in loader:
            src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
            output = model(src_batch, trg_batch, 0.0)
            output = output[:, 1:].reshape(-1, vocab_size)
            target = trg_batch[:, 1:].reshape(-1)
            loss = criterion(output, target)
            epoch_loss += loss.item()

        return epoch_loss / len(loader)

In [92]:
#Find the models accuracy on a Dataloader
def accuracy(model, loader, device):
    with torch.no_grad():
        model.eval()
        accurate = 0
        total = 0

        for src_batch, trg_batch in loader:
            src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
            output = model(src_batch, trg_batch, 0.0)

            output = output[:, 1:].reshape(-1, vocab_size)
            target = trg_batch[:, 1:].reshape(-1)

            #Creates list of booleans, False if 0, else True
            mask = target != pad_val
            preds = output.argmax(dim=1)

            #Ignoring padding tokens when evaluating
            accurate += (preds[mask] == target[mask]).sum().item()
            total += mask.sum().item()

        return accurate/total

In [93]:
def test(model, test_seq):
    with torch.no_grad():
        model.eval()
        encoded = torch.tensor([sos_val] + test_seq + [eos_val], dtype=torch.long)

        #Pad our text to the length the model was trained on
        pad_len = inp_padded.shape[1] - len(encoded)
        encoded = torch.cat([encoded, torch.zeros(pad_len, dtype=torch.long).fill_(pad_val)])

        #Gives it batch size 1
        encoded = encoded.unsqueeze(0).to(device)

        #Context vector
        hidden = model.encoder(encoded)

        predicted = []
        decoder_input = torch.tensor([sos_val], device=device)
        for step in range(len(test_seq)):
            prediction, hidden = model.decoder(decoder_input, hidden)
            best_guess = prediction.argmax(1).item()
            if best_guess == eos_val:
                break

            predicted.append(best_guess)
            decoder_input = torch.tensor([best_guess], device=device)
        return(predicted)

In [94]:
#Training hyperparameters
num_epochs = 50
learning_rate = 1e-3
encoder_embedding_size = 64
decoder_embedding_size = 64
hidden_size = 128
num_layers = 1

#Model hyperparameters
encoder_net = Encoder(vocab_size, encoder_embedding_size, hidden_size, num_layers).to(device)
decoder_net = Decoder(vocab_size, decoder_embedding_size, hidden_size, vocab_size, num_layers).to(device)

model = Seq2Seq(encoder_net, decoder_net).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
#Ignore <pad>
criterion = nn.CrossEntropyLoss(ignore_index=pad_val)
#Combat overfitting
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7
)

In [95]:
test_str = [8, 0, 3, 6, 5]
tf_ratio = 1
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
      for src_batch, trg_batch in train_loader:
          src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)
          optimizer.zero_grad()

          output = model(src_batch, trg_batch, max(0.0, tf_ratio))

          #Skip <sos>, then flatten the batch size and seq length dimensions into 1
          output = output[:, 1:].reshape(-1, vocab_size)
          target = trg_batch[:, 1:].reshape(-1)

          loss = criterion(output, target)
          loss.backward()

          #Norm=magnitude, clipping gradient to ensure they don't explode
          torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1)
          optimizer.step()
          epoch_loss += loss.item()

    print(f"{test_str} => {test(model, test_str)}")
    train_loss = epoch_loss/len(train_loader)
    print(f"Train Loss: {train_loss}")
    val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Validation Loss: {val_loss}")
    tf_ratio -= 0.025
    scheduler.step(val_loss)

Epoch 1 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.7999969902932644
Validation Loss: 0.4442564660594577
Epoch 2 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.1775735115110874
Validation Loss: 0.20152511388536484
Epoch 3 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.10150135255604982
Validation Loss: 0.12589375852119356
Epoch 4 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.0693022271245718
Validation Loss: 0.08039183548045536
Epoch 5 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.05037573451921344
Validation Loss: 0.054224053575169476
Epoch 6 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.0392469498347491
Validation Loss: 0.04100294353529101
Epoch 7 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.03274655818939209
Validation Loss: 0.027029198298733386
Epoch 8 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.02924300869181752
Validation Loss: 0.02847459473248039
Epoch 9 / 50
[8, 0, 3, 6, 5] => [5, 6, 3, 0, 8]
Train Loss: 0.02538

In [96]:
print(f"Train Loss: {evaluate(model, train_loader, criterion, device)}")
print(f"Train Acc: {accuracy(model, train_loader, device)}")
print(f"Val Loss: {evaluate(model, val_loader, criterion, device)}")
print(f"Val Acc: {accuracy(model, val_loader, device)}")
print(f"Test Loss: {evaluate(model, test_loader, criterion, device)}")
print(f"Test Acc: {accuracy(model, test_loader, device)}")

Train Loss: 0.0016325589979533106
Train Acc: 1.0
Val Loss: 0.001731661455114446
Val Acc: 1.0
Test Loss: 0.0017887652511634523
Test Acc: 1.0


In [97]:
torch.save(model.state_dict(), "gru_num_reversal_model.pt")
# model.load_state_dict(torch.load("gru_num_copy_model.pt", weights_only=True, map_location=torch.device('cuda')))

In [98]:
def build_context_df(model, loader, inp_padded, device):
    model.eval()
    all_sources = []
    all_targets = []
    all_contexts = []

    with torch.no_grad():
        for src_batch, trg_batch in loader:
            src_batch = src_batch.to(device)
            trg_batch = trg_batch.to(device)
            hidden = model.encoder(src_batch)

            context = hidden[0]

            # Concatenate along feature dim: (batch_size, 2 * hidden_size)
            # To undo, h_final = context_vector[:, :128], c_final = context_vector[:, 128:]

            all_sources.append(src_batch.cpu())
            all_targets.append(trg_batch.cpu())
            all_contexts.append(context.cpu())

    all_sources = torch.cat(all_sources, dim=0)   # (N, seq_len)
    all_targets = torch.cat(all_targets, dim=0)   # (N, seq_len)
    all_contexts = torch.cat(all_contexts, dim=0)  # (N, 2 * hidden_size)

    source_lists = [row.tolist() for row in all_sources]
    target_lists = [row.tolist() for row in all_targets]
    context_lists = [row.tolist() for row in all_contexts]

    context_df = pd.DataFrame({
        "source": source_lists,
        "context_vector": context_lists,
        "target": target_lists
    })

    return context_df

full_loader = DataLoader(dataset, batch_size=32, shuffle=False)
context_df = build_context_df(model, full_loader, inp_padded, device)

In [99]:
def strip_padding(tokens):
    while tokens and tokens[-1] == pad_val:
        tokens = tokens[:-1]
    return tokens

context_df["source"] = context_df["source"].apply(strip_padding)
context_df["target"] = context_df["target"].apply(strip_padding)

In [100]:
display(context_df)

,source,context_vector,target
0,"[10, 8, 1, 8, 2, 1, 6, 11]","[-0.046401094645261765, -0.9328272342681885, 0...","[10, 6, 1, 2, 8, 1, 8, 11]"
1,"[10, 2, 3, 3, 7, 9, 6, 11]","[0.08333681523799896, -0.9335523843765259, -0....","[10, 6, 9, 7, 3, 3, 2, 11]"
2,"[10, 8, 5, 9, 4, 6, 7, 11]","[0.16618485748767853, -0.9464005827903748, 0.0...","[10, 7, 6, 4, 9, 5, 8, 11]"
3,"[10, 0, 6, 5, 9, 8, 7, 11]","[-0.42709603905677795, -0.9403269290924072, 0....","[10, 7, 8, 9, 5, 6, 0, 11]"
4,"[10, 2, 7, 3, 4, 4, 11]","[-0.3000599145889282, -0.8554231524467468, -0....","[10, 4, 4, 3, 7, 2, 11]"
...,...,...,...
79989,"[10, 4, 7, 0, 6, 4, 3, 7, 2, 11]","[-0.3555837571620941, -0.975588321685791, 0.30...","[10, 2, 7, 3, 4, 6, 0, 7, 4, 11]"
79990,"[10, 2, 3, 5, 3, 5, 0, 5, 1, 9, 11]","[0.3924708068370819, -0.972129225730896, 0.137...","[10, 9, 1, 5, 0, 5, 3, 5, 3, 2, 11]"
79991,"[10, 2, 1, 7, 7, 9, 6, 5, 11]","[-0.07323447614908218, -0.9699262380599976, -0...","[10, 5, 6, 9, 7, 7, 1, 2, 11]"
79992,"[10, 3, 1, 8, 2, 4, 6, 2, 8, 11]","[0.289939820766449, -0.9745516777038574, -0.31...","[10, 8, 2, 6, 4, 2, 8, 1, 3, 11]"


In [101]:
context_df.to_csv("gru_reversal_data.csv")